# Datenbereinigung

## Erste Schritte der Datenbereinigung

Wie inzwischen wohl schon üblich, importieren wir zuerst die Bibliotheken, die wir am häufigsten brauchen.

In [ ]:
import numpy as np
import pandas as pd

Laden wir die Daten aus `seattle-weather_raw.csv` in einen pandas-DataFrame. Vielleicht kennst du diesen Datensatz schon aus den Notebooks mit den pandas-Übungen. Dort haben wir ihn dir bereits vorverarbeitet zur Verfügung gestellt. Jetzt ist es die Rohdatei, und wir müssen uns Gedanken machen, wie wir sie bereinigen.

In [ ]:
# Rohdaten importieren
df_weather = pd.read_csv("../data/seattle-weather_raw.csv")

> **Übung:**
>
> Verschaffe dir einen schnellen Überblick über die Daten. Nutze die bekannten pandas-Methoden wie `.head()`, `.tail()`, `.info()` und `.describe()`. Fällt dir etwas auf? Was würdest du bei der Datenbereinigung angehen?

<details><summary>
Hier klicken für einen Hinweis…
</summary>
Siehst du:
  
    - Formatierungsprobleme in den Spaltennamen?
    - Duplikate?
    - fehlende Werte? (Was ist der Grund? Und was kannst du dagegen tun?)
    - Inkonsistenzen in der Bedeutung der Daten?
    - unterschiedliche Einheiten?
    - andere Datentypen als erwartet?
    - offensichtliche Ausreißer / unerwartete Werte?
</details>


In [ ]:
# dein Code: Datentypen und statistische Übersicht der DataFrame-Spalten prüfen
df_weather.info()
df_weather.describe()

**Erklärung**

`.info()` gibt die Spaltennamen, die Anzahl der Nicht-Null-Werte und die dtypes aus. Dadurch fallen sofort die object-Spalten auf (date, precipitation, temp_min, weather), die eigentlich numerisch bzw. vom Typ datetime sein sollten. `.describe()` ergänzt Kennzahlen für die numerischen Spalten. Dass temp_min dort einen höheren Mittelwert als temp_max hat, ist ein Hinweis auf die vermischten Einheiten Fahrenheit und Celsius. Zusammen mit dem `.head()`-Aufruf oben reicht das, um die unordentlichen Spaltennamen, die doppelten Zeilen, die fehlenden Werte sowie die Einheiten- und Typprobleme zu erkennen, die im weiteren Verlauf des Notebooks behoben werden.


In [ ]:
df_weather.head(7)

Los geht's!

## Format der Spaltennamen korrigieren

Wie dir wahrscheinlich schon aufgefallen ist, müssen wir die Spaltennamen ändern. Wie bereits in früheren Notebooks erklärt, ist es immer eine gute Idee, Leerzeichen durch Unterstriche zu ersetzen. Außerdem empfiehlt es sich, in Spaltennamen nur Kleinbuchstaben zu verwenden. Zudem hat die letzte Spalte keinen Namen.

Eine Möglichkeit, die Namen zu ändern, besteht darin, die aktuellen Namen in eine Liste zu schreiben, sie dort zu bearbeiten und die Liste anschließend wieder dem DataFrame zuzuweisen, wie es in den pandas-Notebooks erklärt wird. Man kann die Spaltennamen aber auch direkt im DataFrame bearbeiten. Letztendlich kannst du natürlich den Weg wählen, mit dem du dich am wohlsten fühlst.

Mit dem `.str`-Accessor kannst du String-Funktionen auf eine DataFrame-Spalte anwenden. Damit können wir Leerzeichen ganz einfach durch Unterstriche ersetzen.

In [ ]:
# Leerzeichen in den Spaltennamen durch Unterstriche ersetzen
df_weather.columns = df_weather.columns.str.replace(" ", "_")
df_weather.columns

Auf die gleiche Weise können wir unserem DataFrame einfach sagen, dass die Spaltennamen nur Kleinbuchstaben enthalten sollen.

In [ ]:
# alle Spaltennamen in Kleinbuchstaben umwandeln
df_weather.columns = df_weather.columns.str.lower()
df_weather.columns

Wir könnten `.str` auch verwenden, um die letzte Spalte umzubenennen. pandas hat dafür aber bereits die Funktion `.rename` eingebaut. So können wir die Spaltennamen direkt ansprechen und brauchen nur ein Dictionary, um die Namen zu ändern.

In [ ]:
# letzte Spalte umbenennen
df_weather = df_weather.rename({"-": "weather"}, axis=1)
df_weather.columns

Glückwunsch! Du hast gerade den ersten Schritt der Bereinigung dieses DataFrames abgeschlossen. Schauen wir uns die ersten Zeilen noch einmal an. Was fällt dir sonst noch Seltsames auf?

In [ ]:
df_weather.head(5)

## Umgang mit Duplikaten

Die Zeilen mit Index $2$ und $3$ sind identisch. Offensichtlich gibt es Duplikate, was zum Beispiel beim Zusammenführen zweier DataFrames passieren kann. Die Funktion `.duplicated()` gibt eine boolesche pandas-Series zurück, die angibt, ob eine Zeile ein Duplikat ist oder nicht. In Kombination mit einer weiteren bekannten Funktion erfahren wir, wie viele doppelte Zeilen wir haben.

In [ ]:
# prüfen, wie viele doppelte Zeilen im DataFrame existieren
df_weather.duplicated().value_counts()

Wie du siehst, haben wir 37 doppelte Zeilen. Diese können wir einfach entfernen. Außerdem müssen wir den alten Index verwerfen und neu erstellen.

In [ ]:
# Duplikate entfernen
df_weather = df_weather.drop_duplicates()
# Index direkt (inplace) zurücksetzen
df_weather.reset_index(inplace=True, drop=True)
df_weather.head(5)

## Datentypen und Datentransformation

Ein Teil der Datenbereinigung besteht darin, die Datentypen in die richtige Form zu bringen. Das ist wichtig, damit wir später besser mit den Daten arbeiten können. Der dtype `object` bedeutet in der Regel, dass die Spalte Strings oder mehrere verschiedene Datentypen enthält.

In [ ]:
# Datentypen im DataFrame prüfen
df_weather.dtypes

Gehen wir die Spaltenliste durch und beginnen vorne mit der Spalte **date**. Momentan hat sie den Datentyp `object`. Wir schauen uns genauer an, als welchen Datentyp pandas unseren aktuellen Datumseintrag erkennt. Dazu nehmen wir einfach den ersten Eintrag.

In [ ]:
# Typ des ersten Datumseintrags
type(df_weather["date"][0])

Wie du siehst, ist unser Datumseintrag nur ein String. Den können wir ganz einfach in ein Datetime umwandeln, wie wir es bereits gelernt haben. Beachte, dass du das auch schon beim Einlesen der CSV-Datei mit dem Parameter `parse_dates=['date']` hättest erledigen können.

In [ ]:
# dtype von "date" in datetime mit dem Format %Y/%m/%d umwandeln
df_weather["date"] = pd.to_datetime(df_weather["date"], format="%Y/%m/%d")

Eine kurze Prüfung, ob das funktioniert hat:

In [ ]:
# dtype von "date" prüfen
type(df_weather["date"][0])

Weiter geht's mit der Spalte **precipitation** (Niederschlag). Beim Blick auf den Kopf des DataFrames hätten wir den Datentyp float erwartet. Das ist ein Hinweis darauf, dass diese Spalte mehr als nur Zahlen enthält. Wir könnten trotzdem einfach versuchen, die Spalte in float umzuwandeln – auch wenn das möglicherweise nicht klappt.

In [ ]:
# Datentyp in float umwandeln
df_weather = df_weather.astype({"precipitation": float})

Wie erwartet erhalten wir eine Fehlermeldung. Sie kann uns aber helfen herauszufinden, warum sich unsere Spalte nicht in den Datentyp `float` umwandeln lässt. Die letzte Zeile der Meldung sagt uns, dass ein String nicht in einen Float umgewandelt werden kann. Damit kennen wir auch die Ursache des Fehlers: Irgendwo in dieser Spalte versteckt sich ein `$`-Zeichen. Aus Neugier versuchen wir, es zu finden.

In [ ]:
# Zeilen mit "$" in der Spalte precipitation anzeigen
df_weather.query('precipitation == "$"')

Es gibt nur ein einziges `$`-Zeichen, in Zeile $1130$. Wir können es einfach durch einen NumPy-NaN-Wert ersetzen. NaN steht für „Not a Number“ und dient als Platzhalter für fehlende Werte. Anschließend können wir den Datentyp der Spalte umwandeln.

In [ ]:
# das `$`-Zeichen durch einen NumPy-NaN-Wert ersetzen
df_weather["precipitation"] = df_weather.precipitation.replace("$", np.nan)
# Datentyp in float umwandeln
df_weather = df_weather.astype({"precipitation": float})
df_weather.precipitation.dtypes

Wenn wir uns die Datentypen und den Tabellenkopf noch einmal ansehen, sieht die Spalte `temp_max` gut aus. Bei `temp_min` fallen uns jedoch zwei Dinge auf: Erstens enthält `temp_min` nicht nur Zahlen (der dtype wird als object erkannt), und zweitens ist die Minimaltemperatur höher als die Maximaltemperatur. Wir können davon ausgehen, dass die Temperatureinheiten durcheinandergeraten sind: Die Maximaltemperatur ist in Celsius angegeben, die Minimaltemperatur offenbar in Fahrenheit. Um das zu beheben, entfernen wir zuerst die Einheit aus den Werten und ändern den Datentyp.

In [ ]:
# Einheit aus der Spalte "temp_min" entfernen, dtype in float umwandeln
df_weather["temp_min"] = df_weather.temp_min.str.strip(" F").astype("float")
df_weather.temp_min.dtype

Jetzt, da unsere Spalte Floats enthält, können wir die Temperatur von Fahrenheit in Celsius umrechnen.

In [ ]:
# temp_min in Celsius umrechnen
df_weather["temp_min"] = df_weather.temp_min.apply(lambda x: (x - 32) / 1.8)
df_weather.head(5)

Das sieht schon viel besser aus. Außerdem ist die Minimaltemperatur jetzt niedriger als die Maximaltemperatur.

Bei der Spalte **wind** ist der Datentyp bereits so, wie er sein sollte. Trotzdem solltest du diese Zahlen für die weitere Arbeit runden. Ein paar Nachkommastellen können für die spätere Modellierung sinnvoll sein, aber so genau muss es nicht immer sein. Probieren wir das Runden also zu Übungszwecken aus.

In [ ]:
# Winddaten runden
df_weather["wind"] = df_weather.wind.round(2)
df_weather.head(5)

Die letzte Spalte enthält keine Zahl, sondern einen String, der die Wetterlage beschreibt. Solche Daten nennt man kategoriale Werte. Wir können herausfinden, wie viele verschiedene kategoriale Werte es gibt.


In [ ]:
# Anzahl der unterschiedlichen Werte anzeigen
df_weather.weather.nunique()

Das sind ziemlich viele – und, wie dir vielleicht schon aufgefallen ist, nicht korrekt. Es gibt mehrere Einträge, die dasselbe bedeuten.

In [ ]:
# alle unterschiedlichen Werte anzeigen
df_weather["weather"].unique()

Bei den eindeutigen Werten sehen wir, dass "drizzle", "Drizzle", "d" und "driz." vermutlich alle dieselbe Kategorie darstellen. Eine Kategorie hat also mehrere Werte. Das passiert häufig, wenn es keinen festen Standard für die Datenerhebung gibt. Wir müssen die Werte jeder Kategorie auf einen einzigen reduzieren. Der einfachste und sicherste Weg ist, jeden Wert manuell der passenden Kategorie zuzuordnen. Die eingebaute Funktion `.map` verwendet dafür entweder ein Dictionary, das jedem Wert seinen neuen Wert zuordnet, oder eine Funktion, die auf jeden Wert angewendet wird. In unserem Fall verwenden wir das Dictionary.

> [!NOTE]
> Manchmal sind Abkürzungen mehrdeutig. Zum Beispiel gibt es die Kategorie "sn.", die entweder für "snow" oder für "sun" stehen könnte. Hier lohnt es sich, den Prozess der Datenerhebung genauer zu untersuchen, um eine fundiertere Entscheidung treffen zu können.
> Du könntest diese Aufgabe auch mit Pattern Matching (über reguläre Ausdrücke) oder Fuzzy Matching angehen. Diese Ansätze sind aber aufwendiger zu programmieren und setzen ein gründliches Verständnis voraus, da die Gefahr besteht, einen Wert der falschen Kategorie zuzuordnen.

In [ ]:
# Anzahl der kategorialen Werte durch Mapping mit einem Dictionary reduzieren
df_weather["weather"] = df_weather["weather"].map(
    {
        "drizzle": "drizzle",
        "r": "rain",
        "r.": "rain",
        "rain": "rain",
        "s": "sun",
        "Rain": "rain",
        "sun": "sun",
        "Sun": "sun",
        "Snow": "snow",
        "sn.": "snow",
        "sw": "snow",
        "d": "drizzle",
        "s.": "sun",
        "driz.": "drizzle",
        "snow": "snow",
        "Drizzle": "drizzle",
        "Fog": "fog",
        "f": "fog",
        "f.": "fog",
        "fog": "fog",
    }
)
df_weather.weather.unique()

Den Datentyp dieser Spalte können wir genauso einfach wie zuvor in `string` umwandeln. Beachte, dass das für die weitere Arbeit nicht zwingend nötig ist.

In [ ]:
# in dtype "string" umwandeln
df_weather["weather"] = df_weather.weather.astype("string")
df_weather.weather.dtype

## Fehlende Daten und Imputation

Fehlende Daten sind ein häufiges Problem bei der Arbeit mit echten Datensätzen. Die Gründe dafür können vielfältig sein, z. B. Sensorausfälle, alte Daten, unsachgemäßes Datenmanagement oder auch menschliche Fehler.

Es ist wichtig, fehlende Daten vor der weiteren Datenanalyse oder dem maschinellen Lernen zu erkennen und angemessen zu behandeln. Viele Machine-Learning-Algorithmen können nicht mit fehlenden Daten umgehen und erfordern, dass ganze Zeilen oder Spalten gelöscht werden, in denen auch nur ein einziger Wert fehlt. Alternativ müssen die fehlenden Daten durch einen neuen Wert ersetzt (imputiert) werden.

Zu wissen, wann fehlende Daten auftreten, kann helfen, fehlende Werte künftig zu vermeiden oder besser mit ihnen umzugehen.
Es gibt eine großartige Python-Bibliothek speziell für die EDA fehlender Daten: **[missingno](https://github.com/ResidentMario/missingno)**. (Nicht zu verwechseln mit dem gleichnamigen Pokémon.)

In [ ]:
# missingno importieren
import missingno as msno

Schauen wir uns an, wie viele Werte pro Spalte in unseren Daten fehlen.

In [ ]:
# Anzahl fehlender Werte pro Spalte anzeigen
df_weather.isna().sum()

Mit missingno können wir das ganz einfach als Balkendiagramm darstellen. Jeder Balken steht für eine Spalte. Auf der linken Seite sehen wir den prozentualen Anteil der vorhandenen Daten, auf der rechten Seite die genaue Anzahl der Werte, die diesen Anteilen entspricht.

In [ ]:
# Anteil vorhandener bzw. fehlender Werte pro Spalte plotten
msno.bar(df_weather)

Wir sehen, dass nur zwei Spalten fehlende Werte enthalten: In der Spalte "temp_max" fehlen mehr als 20 % der Daten, in der Spalte "precipitation" weniger als 10 %. Interessant wäre, ob es ein Muster beim Auftreten der fehlenden Werte gibt. Das würde uns helfen zu verstehen, woher sie kommen. Es könnte zum Beispiel sein, dass diese Werte sonntags nicht erhoben werden. Dann hätten wir in jeder 7. Zeile einen fehlenden Wert. Es könnte auch auf einen Zeitraum hinweisen, in dem unser Messgerät ausgefallen ist, was zu aufeinanderfolgenden fehlenden Werten führen würde. All das hilft uns, unsere Daten besser zu verstehen und Fehler gegebenenfalls zu korrigieren.

Mit missingno kannst du auch eine Matrix-Darstellung erzeugen. Für jede Zeile im Datensatz siehst du, ob in den einzelnen Spalten Daten fehlen. Vorhandene Daten werden grau (oder in einer Farbe deiner Wahl) dargestellt, fehlende Daten weiß. So bekommst du einen Eindruck davon, wie sich die fehlenden Daten im DataFrame verteilen und ob sie gemeinsam auftreten.

In [ ]:
msno.matrix(df_weather)

Auf der rechten Seite siehst du für jeden Datenpunkt, wie viele Spalten Daten enthalten. In unserem Fall reicht die Spanne von 4 Spalten mit Daten (links) bis 6 Spalten mit Daten (rechts). Wie du siehst, gibt es nur wenige Zeilen, in denen fehlende Werte gemeinsam auftreten. Insgesamt ist kein klares Muster in den fehlenden Daten erkennbar. Wir müssen daher annehmen, dass die fehlenden Daten zufällig auftreten. Das hilft uns leider überhaupt nicht bei der Frage, wie wir diese fehlenden Werte am besten behandeln. 

missingno bietet noch weitere Diagramme zur Untersuchung fehlender Werte. Du kannst zum Beispiel auch eine Heatmap erstellen, um zu sehen, wie oft fehlende Werte in bestimmten Features gemeinsam auftreten. Schau dir das Repository für weitere Einblicke an!

### Imputation von Daten

Fehlende Daten und Imputation sind große Themen in der Data Science. Imputieren bedeutet, fehlende Werte durch sinnvolle Werte zu ersetzen. Es gibt verschiedene Imputationstechniken, mit denen wir uns in diesem Notebook vertraut machen. Wenn du fehlende Werte imputierst, manipulierst du die Daten, und das kann sich erheblich auf die Datenqualität auswirken. Du brauchst daher ein gewisses Wissen über deine Daten, um eine gute Lösung und den für die jeweilige Aufgabe am besten geeigneten Ansatz zu finden. Wie so oft in der Data Science gibt es keine Lösung, die für alle Probleme passt. Die Imputation reicht von recht einfachen bis zu hochkomplexen Strategien. Denk daran, dass die Arbeit in der Data Science in Zyklen verläuft. Am besten beginnst du immer mit einem einfachen Ansatz und wirst mit jedem Zyklus komplexer. So wird deine Vorverarbeitung nur dann komplexer und damit zeitaufwendiger, wenn das tatsächlich einen Mehrwert bringt.

Der erste Schritt ist immer festzustellen, wie viele Daten fehlen. Da wir oben bereits alle fehlenden Daten identifiziert haben, brauchen wir dafür nur einen Befehl.

> **Übung:** Zeit für ein Brainstorming! Was könnten Gründe für fehlende Werte sein, und welche Möglichkeiten zum Umgang mit fehlenden Daten fallen dir ein? Hängen diese Möglichkeiten von den Gründen für die fehlenden Werte ab?

**Erklärung**

Fehlende Werte entstehen aus vielen Gründen: Ausfall von Sensoren oder Messgeräten, Daten, die an bestimmten Tagen nicht erhoben werden, leer gelassene Felder in Formularen, Fehler beim Zusammenführen oder Exportieren oder Werte, die auf einen Datensatz schlicht nicht zutreffen. In der Statistik unterscheidet man drei Mechanismen: MCAR (missing completely at random, vollständig zufällig fehlend), MAR (missing at random, zufällig fehlend, erklärbar durch andere beobachtete Spalten) und MNAR (missing not at random, nicht zufällig fehlend, wobei das Fehlen vom nicht beobachteten Wert selbst abhängt). Zum Umgang damit gehören das Entfernen von Zeilen oder Spalten, die Imputation mit einer Konstante oder mit Mittelwert, Median oder Modus, Forward bzw. Backward Fill, Interpolation und die Vorhersage des Werts mit einem Modell. Die richtige Wahl hängt tatsächlich vom Mechanismus ab: Bei MCAR und einem kleinen Anteil können das Entfernen oder eine einfache Mittelwert-Imputation ausreichen. Bei MAR oder MNAR ist eine Methode sicherer, die die zusammenhängenden Spalten nutzt, da eine pauschale Imputation die Daten verzerren würde.


### Daten verwerfen

Das Verwerfen von Daten ist die einfachste und schnellste Option. Es ist aber nur zu empfehlen, wenn von vornherein viele Daten vorhanden sind und der Anteil fehlender Werte gering ist. Es gibt mehrere Möglichkeiten: Du kannst die Zeilen entfernen, die fehlende Werte enthalten, oder sogar eine ganze Spalte.

Bei sehr vielen fehlenden Daten bleiben dir am Ende aber womöglich nur noch wenige Daten übrig. Außerdem könntest du (wertvolle) Informationen verlieren oder sogar die Repräsentation bestimmter Daten verändern, wenn die Zeilen mit fehlenden Werten etwas gemeinsam haben und nicht völlig zufällig sind. Du solltest beim Verwerfen von Daten daher immer umsichtig vorgehen.

In unserem Fall haben wir nur 2 Spalten mit fehlenden Werten, und die Menge scheint nicht allzu groß zu sein. Setzen wir diese Zahlen aber einmal in Relation.

In [ ]:
print(f"Anzahl der Zeilen : {df_weather.shape[0]}")
print(
    f"fehlende Werte in precipitation : {round(df_weather.precipitation.isna().sum() / df_weather.shape[0] * 100, 2)} %"
)
print(
    f"fehlende Werte in temp_max : {round(df_weather.temp_max.isna().sum() / df_weather.shape[0] * 100, 2)} %"
)
print(
    f"fehlende Werte im DataFrame : {round(df_weather.isna().sum().sum() / (df_weather.shape[0] * df_weather.shape[1]) * 100, 2)} %"
)

Wir haben 1461 Zeilen, und etwa ein Viertel davon hat in mindestens einer Spalte fehlende Werte. Das wäre eigentlich zu viel, um es einfach zu verwerfen. Zum Glück gibt es noch weitere Möglichkeiten der Imputation. Für den Anfang ist das Verwerfen aber durchaus eine Option.

pandas hat eine Funktion zum Entfernen fehlender Werte. In der folgenden Tabelle findest du eine Beschreibung einiger Parameter:

`dropna` | Beschreibung
---|---
`.dropna()` | entfernt eine Zeile, wenn sie mindestens 1 NaN enthält
`.dropna(how='all')` | entfernt eine Zeile, wenn alle Einträge NaN sind
`.dropna(thresh=2)` | entfernt eine Zeile, wenn weniger als 2 Zellen einen gültigen Wert enthalten

Schau dir auch die [Dokumentation zu `dropna`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.dropna.html) an.

> **Übung:** Da wir dir in diesem Notebook auch andere Imputationsmethoden zeigen wollen, erstelle eine Kopie unseres Datensatzes. Wende anschließend `.dropna` auf den kopierten Datensatz an. Probiere mindestens zwei verschiedene Varianten aus.


In [ ]:
# eine Kopie des Datensatzes erstellen, damit der bereinigte df_weather unverändert bleibt
df_dropna = df_weather.copy()

# Variante 1: eine Zeile entfernen, wenn sie mindestens einen fehlenden Wert enthält (in einer beliebigen Spalte)
df_dropna_any = df_dropna.dropna()

# Variante 2: eine Zeile nur entfernen, wenn alle ihre Werte fehlen
df_dropna_all = df_dropna.dropna(how="all")

print(f"Zeilen vor dem Entfernen        : {df_weather.shape[0]}")
print(f"Zeilen nach dropna()            : {df_dropna_any.shape[0]}")
print(f"Zeilen nach dropna(how='all')   : {df_dropna_all.shape[0]}")

**Erklärung**

Wir arbeiten mit `.copy()`, damit der bereinigte `df_weather` durch das Experiment nicht verändert wird. `.dropna()` entfernt jede Zeile mit mindestens einem NaN, hier also jede Zeile, in der precipitation oder temp_max fehlt. `.dropna(how="all")` entfernt nur Zeilen, in denen alle Spalten NaN sind, und ändert bei diesem Datensatz daher nichts. Wenn man die Zeilenzahlen nebeneinander ausgibt, wird dieser Unterschied deutlich. Da etwa ein Viertel der Zeilen betroffen ist, würde ein einfaches `.dropna()` zu viele Daten verwerfen. Genau deshalb lohnt es sich, die folgenden Imputationsmethoden auszuprobieren.


### Imputation mit einer Konstante

#### Fehlende Werte markieren

Manchmal kann sogar ein fehlender Wert informativ sein. Wenn du zum Beispiel eine Testperson nach ihrem Geschlecht fragst und nur die Optionen „weiblich“ oder „männlich“ zum Ankreuzen anbietest, die Person sich aber als nicht-binär definiert, kann das zu einem fehlenden Wert führen. Du könntest dann annehmen, dass dieser Wert für `other` oder einen anderen kategorialen Wert steht, und ihn entsprechend zuweisen.
In unserem Fall könnten wir einen eindeutigen Wert zuweisen. Für Niederschlag und Temperatur wäre das ein sehr hoher absoluter Wert.

#### Imputation mit Mittelwert/Median/Modus

Eine beliebte Imputationstechnik ist es, den Mittelwert, Median oder Modus der Spalte zu verwenden. Welchen der drei du wählst, hängt vom jeweiligen Problem ab. Bei einer kategorialen Variable (z. B. Geschlecht: 0 für männlich, 1 für weiblich) wäre der Mittelwert zum Beispiel nicht sinnvoll. Auch bei numerischen Werten kann der Mittelwert gefährlich sein, da Ausreißer einen großen Einfluss haben können. In diesem Fall ist der Median oft die bessere Wahl.
Natürlich kannst du auch Annahmen treffen, zum Beispiel, dass das Wetter heute dasselbe ist wie am Vortag.

> [!TIP]
> Eine Visualisierung kann dir helfen zu entscheiden, welchen Wert du nehmen solltest.

Wie du wahrscheinlich schon vermutet hast, bietet pandas fertige Funktionen für die Imputation:

Methode | Beschreibung
---|---
`df.fillna(0)` | ersetzt alle NaN in df durch einen einzigen Wert
`df.fillna({'precipitation': 0, 'temp_min': 0, 'temp_max': 42, 'wind': 0, 'weather': 'no_weather'})` | ersetzt NaN in bestimmten Spalten durch bestimmte Werte
`df.ffill()` | verwendet den Wert des Vortags bzw. der vorherigen Zeile (Forward Fill)
`df.bfill()` | verwendet den Wert des Folgetags bzw. der nächsten Zeile (Backward Fill)
`df.ffill(axis=1)` | übernimmt Werte aus der vorherigen Spalte
`df.ffill(limit=1)` | fehlen mehrere Werte hintereinander, wird nur die erste Lücke mit dem Wert des Vortags gefüllt, der Rest bleibt NaN

Schau dir auch die [Dokumentation zu `fillna`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.fillna.html) an.

> **Übung:** Erstelle erneut eine Kopie von `df_weather` und fülle die fehlenden Werte. Probiere mindestens zwei verschiedene Methoden aus. Welche Werte wären sinnvoll?

In [ ]:
# eine Kopie des Datensatzes erstellen
df_fillna = df_weather.copy()

# Methode 1: die numerischen Spalten mit ihrem Median füllen (im Gegensatz zum Mittelwert robust gegenüber Ausreißern)
df_fillna["precipitation"] = df_fillna["precipitation"].fillna(
    df_fillna["precipitation"].median()
)
df_fillna["temp_max"] = df_fillna["temp_max"].fillna(df_fillna["temp_max"].median())

# Methode 2: Forward Fill, den Wert des Vortags übernehmen (das Wetter ist von Tag zu Tag ähnlich)
df_ffill = df_weather.copy()
df_ffill = df_ffill.ffill()

print("fehlende Werte nach Auffüllen mit dem Median:")
print(df_fillna.isna().sum())
print("\nfehlende Werte nach Forward Fill:")
print(df_ffill.isna().sum())

**Erklärung**

Bei der Median-Imputation wird jede numerische Spalte mit ihrem eigenen Median aufgefüllt. Der Median ist hier dem Mittelwert vorzuziehen, weil Niederschlag und Temperatur Ausreißer enthalten können, die den Mittelwert verzerren würden. Forward Fill mit `.ffill()` überträgt stattdessen den Wert des Vortags in die Lücke. Das ist bei täglichen Wetterdaten eine vernünftige Annahme, da aufeinanderfolgende Tage ähnlich sind. Wir verwenden `.ffill()` statt des älteren `fillna(method="ffill")`, weil das Argument `method` in neueren pandas-Versionen veraltet ist. Die Spalte weather ist kategorial, ein Median ist dort also bedeutungslos: Hier wären Forward Fill, der Modus oder eine Konstante das richtige Werkzeug.


#### Interpolation

Die letzte Methode, die wir heute ausprobieren wollen, ist die Berechnung fehlender Werte per Interpolation. Eine Interpolation schätzt den Verlauf einer stetigen Funktion zwischen zwei bekannten Punkten. Mithilfe dieser Funktion können wir den fehlenden Wert annähern.

Es gibt verschiedene Arten der Interpolation. Die gebräuchlichste und zugleich einfachste für einen fehlenden Wert $x_{i}$ ist die lineare Interpolation, bei der zwei Datenpunkte, $(x_{i-1},y_{i-1})$ und $(x_{i+1},y_{i+1})$, durch eine Gerade verbunden werden.

$$
y_{i} = y_{i-1} + \frac{y_{i+1} - y_{i-1}}{x_{i+1} - x_{i-1}}(x_i - x_{i-1})
$$

In unserem Fall ist die Maximaltemperatur ein perfektes Beispiel. Mit den Standardeinstellungen von pandas' `.interpolate` erhalten wir eine lineare Interpolation, bei der die Werte als gleichmäßig verteilt angenommen werden. Alternativ kann auch das Datum als Grundlage dienen.

> **Übung:** Erstelle erneut eine Kopie von `df_weather` und versuche, die fehlenden Werte in der Spalte mit der Maximaltemperatur zu interpolieren. Ein Blick in die [Dokumentation zu `interpolate`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.interpolate.html) lohnt sich.

In [ ]:
# eine Kopie des Datensatzes erstellen
df_interpolate = df_weather.copy()

# lineare Interpolation schätzt jeden fehlenden Wert aus den Punkten auf beiden Seiten der Lücke
df_interpolate["temp_max"] = df_interpolate["temp_max"].interpolate()

print(f"fehlende temp_max-Werte vorher : {df_weather['temp_max'].isna().sum()}")
print(f"fehlende temp_max-Werte nachher: {df_interpolate['temp_max'].isna().sum()}")

**Erklärung**

`.interpolate()` verwendet standardmäßig eine lineare Interpolation: Zwischen den bekannten Werten auf beiden Seiten einer Lücke wird eine Gerade gezogen, und der fehlende Wert wird von dieser Geraden abgelesen. Das passt gut zu temp_max, weil sich die Temperatur allmählich ändert und die Zeilen nach Datum sortiert sind, sodass benachbarte Tage gute Schätzwerte liefern. Eine Lücke ganz am Anfang der Spalte kann so nicht gefüllt werden, da es keinen früheren Punkt gibt, von dem aus interpoliert werden könnte. Ein führendes NaN kann also bestehen bleiben. Bei ungleichmäßigen Abständen könntest du das Datum als Index setzen und `method="time"` übergeben. Dann wird die Schätzung danach gewichtet, wie weit die umliegenden Daten zeitlich auseinanderliegen.


Da auch Splines und Polynome verwendet werden können, ist die Interpolationsfunktion ziemlich mächtig. Um sie sinnvoll einzusetzen, ist es unbedingt notwendig, die weiterführende Dokumentation zu lesen!

### Vorhersage

Es ist auch möglich, fehlende Werte vorherzusagen. Dabei wird die Spalte mit den fehlenden Daten zur Zielvariablen. Der Datensatz wird aufgeteilt in einen Teil mit gültigen Zielwerten, der zum Training verwendet wird, und einen Teil mit den fehlenden Werten, für den wir Vorhersagen treffen. Ein gutes Modell für die Imputation ist KNN. Da wir das Thema prädiktive Modellierung noch nicht behandelt haben, wird es an dieser Stelle nur erwähnt.

> [!NOTE]
> Es ist wichtig, jegliches Data Leakage zu vermeiden! Wenn du ein Modell zur Imputation fehlender Daten verwendest, stelle sicher, dass du deine eigentlichen Zielwerte entfernst! Wenn du temp_max imputierst, darfst du die Spalte weather nicht verwenden, wenn dein eigentliches Ziel die Vorhersage des Wetters ist.